In [4]:
import anthropic
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PayloadSchemaType, PointStruct, SparseVectorParams, Document, Prefetch, FusionQuery
from qdrant_client import models
import pandas as pd
import fastembed

In [2]:
qdrant_client = QdrantClient(url="http://localhost:6333")

### New qdrant collection for hybrid search

In [19]:
qdrant_client.create_collection(
        collection_name="Amazon-items-collection-01-hybrid-search",
        vectors_config={
                "voyage-3": VectorParams(size=1024, distance=Distance.COSINE)
        },
        sparse_vectors_config={
                "bm25": SparseVectorParams(modifier=models.Modifier.IDF)
        }
) 

True

### 1000 items dataset

In [ ]:
# therefore we need index
qdrant_client.create_payload_index(
        collection_name="Amazon-items-collection-01-hybrid-search",
        field_name="parent_asin",                 # we will be running many queries
        field_schema=PayloadSchemaType.KEYWORD,   # This will be an exact match
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

In [36]:
from dotenv import load_dotenv
import os
import voyageai

load_dotenv()
VOYAGE_API_KEY = os.environ.get("VOYAGE_API_KEY")
vo = voyageai.Client(api_key=VOYAGE_API_KEY)

def get_embedding(text, model='voyage-3', input_type="document"):
        result = vo.embed(
                [text],
                model=model,
                input_type=input_type
        )
        return result.embeddings[0]

def get_embedding_batch(text_list, model='voyage-3', batch_size=100, input_type="document"):
        embeddings = []
        for i in range(0, len(text_list), batch_size):
                batch = text_list[i:i + batch_size]
                result = vo.embed(batch, model=model, input_type=input_type)
                embeddings.extend(result.embeddings)
        return embeddings


In [15]:
# process and embed
def preprocess_desciption(row):
        return f"{row['title']} {' '.join(row['features'])}"


def extract_first_large_image(row):
        return row['images'][0].get('large', '')



df_items = pd.read_json("../../data/meta_Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", lines=True)
df_items["description"] = df_items.apply(preprocess_desciption, axis=1)
df_items["image"] = df_items.apply(extract_first_large_image, axis=1)


data_to_embed = df_items[['description', 'image', 'rating_number', 'price', 'average_rating', 'parent_asin']].to_dict(orient="records")
text_to_embed = [data['description'] for data in data_to_embed]

In [16]:
text_to_embed

['80x100 Monocular-Telescope Low Night Vision Monoculars High Definition for Adults High Powered with Smartphone Adapter Monocular Telescope Hunting Wildlife Bird Watching Travel Camping Hiking ',
 "SoundPEATS Air Conduction Headphones, RunFree Lite Open-Ear Sports Bluetooth V5.3 Headphones with Bass Boost and 17 Hours, Sweatproof Ultralight Headset for Running, Workouts, Comfortable Fit, USB-C 𝗨𝗻𝗯𝗲𝗮𝘁𝗮𝗯𝗹𝗲 𝗖𝗼𝗺𝗳𝗼𝗿𝘁 𝗮𝗻𝗱 𝗟𝗶𝗴𝗵𝘁𝘄𝗲𝗶𝗴𝗵𝘁 𝗗𝗲𝘀𝗶𝗴𝗻 - 🍀 𝘽𝙚𝙨𝙩 𝙑𝙖𝙡𝙪𝙚 𝙊𝙥𝙚𝙣-𝙚𝙖𝙧 𝙎𝙥𝙤𝙧𝙩 𝙃𝙚𝙖𝙙𝙥𝙝𝙤𝙣𝙚 - 𝘾𝙉𝙀𝙏 🍀. The RunFree Lite headset is carefully optimized for weight distribution, ensuring a secure and comfortable fit that eliminates worries of slipping and shaking. Wrapped in skin-friendly liquid silicone, these lightweight headphones (0.99oz) provide a weightless and comfortable experience for extended listening during gym sessions, exercise, running, and more. 𝗢𝗽𝗲𝗻-𝗘𝗮𝗿 𝗗𝗲𝘀𝗶𝗴𝗻 𝗮𝗻𝗱 𝗗𝘆𝗻𝗮𝗺𝗶𝗰 𝗦𝗼𝘂𝗻𝗱 - Stay connected and aware of your surroundings with our open-ear design sports headphones, which use air conductio

In [17]:
embeddings = get_embedding_batch(text_to_embed)

Processed 100 of 1000
Processed 200 of 1000
Processed 300 of 1000
Processed 400 of 1000
Processed 500 of 1000
Processed 600 of 1000
Processed 700 of 1000
Processed 800 of 1000
Processed 900 of 1000
Processed 1000 of 1000


In [ ]:
len(embeddings)

1000

In [20]:
pointStructs = []
i = 1
for embedding, data in zip(embeddings, data_to_embed):
       pointStructs.append(
                PointStruct(
                        id=i,
                        vector={
                                "voyage-3": embedding,
                                "bm25": Document(
                                        text=data['description'],
                                        model="qdrant/bm25"
                                )
                        },
                        payload=data
                )
       )
       i+=1

In [21]:
qdrant_client.upsert(
        collection_name = 'Amazon-items-collection-01-hybrid-search',
        wait = True,
        points = pointStructs,
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

### Hybrid Retrieval

In [35]:
def retrieve_data(query, qdrant_client, k=5):
        query_embedding = get_embedding(query)
        results = qdrant_client.query_points(
                collection_name="Amazon-items-collection-01-hybrid-search",
                prefetch=[
                        Prefetch(
                                query=query_embedding,
                                using='voyage-3',
                                limit=10,
                        ),
                        Prefetch(
                                query=Document(
                                        text=query,
                                        model="qdrant/bm25"
                                ),
                                using='bm25',
                                limit=10,
                        ),
                ],
                query=FusionQuery(fusion='rrf'), #reciprocal rank fusion
                limit=k,
        )

        retrieved_context_ids = []
        retrieved_context = []
        similarity_scores = []
        retrieved_context_ratings = []

        for item in results.points:
                retrieved_context_ids.append(item.payload['parent_asin'])
                retrieved_context.append(item.payload['description'])
                retrieved_context_ratings.append(item.payload['average_rating'])
                similarity_scores.append(item.score)
        
        return {
                "retrieved_context_ids": retrieved_context_ids,
                "retrieved_context": retrieved_context,
                "retrieved_context_ratings": retrieved_context_ratings,
                "similarity_scores": similarity_scores,
        }

In [37]:
retrieve_data("Can we get tablet?", qdrant_client, 20)

{'retrieved_context_ids': ['B09V28Y3HH',
  'B08JCX7MB7',
  'B09Z6F54Y3',
  'B0BN58Z4YX',
  'B0BLMWXBDY',
  'B09PL3WQHB',
  'B0BK91PLJG',
  'B0BLSRBKY7',
  'B0BKPTRBNT',
  'B09ZJ9LKCN',
  'B0BGP3P33P',
  'B09VBQG9ZX',
  'B0B4WNFTVZ',
  'B0B4C98K37',
  'B0B3MTQHD4',
  'B0B87XZT6M',
  'B0C1NM3S2D',
  'B09WCL9HRK',
  'B0BDK85Z37',
  'B0C7GLKTC3'],
 'retrieved_context': ['CIYUPE Tablet Pillow Stand, Tablet Holder Dock for Bed with 6 Viewing Angles, Compatible with iPad Pro 9.7, 10.5,12.9 Air Mini 4 3, Kindle, Galaxy Tab, E-Reader and Books (Grey) 【 COMPATIBILITY 】 Universal tablet stand pillow suitable to iPad and tablets between 4.7 and 13 inches, like new iPad Air 4, 2021 iPad Pro 11, 2020 iPad Pro 11 / 12.9 inch, 2018 iPad Pro 10.5 inch, iPad Air, iPad Mini, Kindle Fire HD 7 8 10, E-reader, iPhone X, iPhone 8 plus, iPhone 13, iPhone 13 Pro Max, iPhone 12 Pro Max, iPhone 11, Surface Pro, Galaxy Tab, Switch. 【6 Viewing Angles】 This multi angle pillow i pad tablet stand has 6 angle adjustme